# 07 — Handling Class Imbalance
3-way comparison: Plain LR vs class_weight="balanced" vs SMOTE.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from src.preprocess import build_preprocessor
from src.config import RANDOM_STATE

In [ ]:
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('data/processed/y_test.csv').values.ravel()

## 1. Why Imbalance Matters
- Dataset: 88% No Default, 12% Default
- A model predicting all-0 gets 88% accuracy but catches zero defaulters
- We need to help the model pay more attention to the minority class

## 2. Method A: class_weight="balanced"

In [ ]:
lr_weighted = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))
])
lr_weighted.fit(X_train, y_train)
y_pred_w = lr_weighted.predict(X_test)
print(classification_report(y_test, y_pred_w, target_names=['No Default', 'Default']))

## 3. Method B: SMOTE
**Critical:** SMOTE must ONLY be applied on training data.
We use `imblearn.pipeline.Pipeline` (NOT sklearn.pipeline.Pipeline)
because sklearn Pipeline does NOT apply sampling steps correctly.

In [ ]:
smote_pipeline = ImbPipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
smote_pipeline.fit(X_train, y_train)
y_pred_s = smote_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_s, target_names=['No Default', 'Default']))

## 4. Verify Test Set Was NOT Resampled

In [ ]:
print(f"X_test shape: {X_test.shape} — unchanged after SMOTE pipeline")
print(f"y_test distribution: {np.bincount(y_test)}")
print("✓ Test set was NOT resampled — SMOTE only affected training")

## 5. Three-Way Comparison

In [ ]:
# Plain LR
lr_plain = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
lr_plain.fit(X_train, y_train)
y_pred_p = lr_plain.predict(X_test)

results = pd.DataFrame({
    'Model': ['Plain LR', 'LR + class_weight', 'LR + SMOTE'],
    'F1 (class 1)': [
        f1_score(y_test, y_pred_p, pos_label=1),
        f1_score(y_test, y_pred_w, pos_label=1),
        f1_score(y_test, y_pred_s, pos_label=1),
    ]
})
print(results.to_string(index=False))